In [1]:
import numpy as np
import torch
import scipy
import pandas as pd
import matplotlib.pyplot as plt
from icp import Inductive_Conformal_Predcition as ICP
from distributions.gaussian_distribution import GaussianUncertainty
from distributions.bingham_distribution import BinghamDistribution
from nc_score import pose_err
from tqdm import tqdm
import tools
# import rpmg
import argparse
from dataset.CameraPoseDataset import CameraPoseDatasetPred
from skimage.io import imread
import cv2
# from Keypoint.ALIKED import aliked_kpts
import torch.nn.functional as F

In [15]:
cal_labels_file = "/Users/runyi/School/Project/TBCP6D/results/PoseNet_results/PhotoTourism/notre_dame_front_facade_val.csv_results.csv"
test_labels_file = "/Users/runyi/School/Project/TBCP6D/results/PoseNet_results/PhotoTourism/notre_dame_front_facade_test.csv_results.csv"

In [16]:
cal_set = CameraPoseDatasetPred(cal_labels_file)
test_set = CameraPoseDatasetPred(test_labels_file)

In [17]:
cal_set.trans_err.mean(), cal_set.rot_err.mean(), test_set.trans_err.mean(), test_set.rot_err.mean()

(0.5770504961532493,
 2.1077664341035396,
 0.5809404071079205,
 1.8159542893645748)

In [24]:
cal_poses = torch.tensor(cal_set.poses)
cal_pred_poses = torch.tensor(cal_set.est_poses)

In [28]:
cal_set.trans_err, cal_set.rot_err.squeeze()

(array([0.28046041, 0.15807435, 0.33208156, ..., 0.87219178, 0.06652045,
        0.98911875]),
 array([1.19504786, 0.52377336, 1.16927007, ..., 0.89118451, 0.46216898,
        0.95647153]))

In [21]:
test_set.trans_err[:10], test_set.rot_err[:10].squeeze()

(array([0.0838365 , 0.27384837, 0.76503871, 0.38461499, 0.48635215,
        3.09940741, 0.01620612, 1.51945531, 0.08644542, 0.31320642]),
 array([1.12224788, 2.44415594, 2.01898497, 0.52487792, 1.83241793,
        2.99738545, 0.30129786, 8.79268038, 1.23430425, 1.8990104 ]))

In [35]:
test_poses = torch.tensor(test_set.poses[:10])
test_est_poses = torch.tensor(test_set.est_poses[:10])

test_poses, test_est_poses

(tensor([[-3.3336e-01,  1.4103e+00,  5.0830e+00,  9.9545e-01,  2.9042e-02,
           8.8447e-02, -2.0464e-02],
         [ 1.3101e+00,  8.4298e-01,  4.4442e+00,  9.9603e-01,  6.8082e-02,
          -5.1076e-02,  2.5895e-02],
         [-8.3682e-01,  7.3251e-01,  2.4081e+00,  9.7357e-01,  6.9592e-03,
           2.1123e-01, -8.6531e-02],
         [-2.1657e-02,  7.5317e-01,  2.7072e+00,  9.9745e-01,  2.9939e-02,
           6.3955e-02, -1.0288e-02],
         [-5.8247e-01, -1.0829e+00, -1.8167e+00,  9.9199e-01, -9.3229e-02,
           7.4817e-02, -4.0926e-02],
         [-1.7865e+00,  1.6341e+00,  6.0280e+00,  9.8686e-01,  3.4187e-02,
           1.4785e-01, -5.5482e-02],
         [ 4.9941e-01, -1.5984e-01, -4.7150e-01,  9.9942e-01,  1.1389e-02,
           3.1649e-02, -5.5526e-03],
         [ 5.1770e-01, -1.3501e+00, -6.9424e-01,  9.9060e-01, -1.2876e-01,
           4.6228e-02, -1.1026e-03],
         [-1.9105e+00,  3.4199e-01,  1.4806e+00,  9.8012e-01,  4.0754e-02,
           1.8805e-01, -4.837

In [47]:
test_trans = test_poses[:, :3]
test_rot = test_poses[:, 3:]
test_est_trans = test_est_poses[:, :3]
test_est_rot = test_est_poses[:, 3:]

In [25]:
icp = ICP(cal_poses, cal_pred_poses, mode="Trans")

In Conformal mode of  Trans


In [33]:
nc_scores = icp.compute_non_conformity_scores()['Trans']

In [41]:
test_nc_scores = pose_err(test_poses, test_est_poses)[0]
test_nc_scores

tensor([0.0838, 0.2738, 0.7650, 0.3846, 0.4864, 3.0994, 0.0162, 1.5195, 0.0864,
        0.3132], dtype=torch.float64)

In [64]:
for i in range(len(test_poses)):
    test_cal_nc_score = icp.trans_err(test_est_trans[i].repeat(len(cal_poses), 1), icp.gt_trans)
    
    p_values = (test_cal_nc_score <= nc_scores).float().mean()
    print(p_values)

tensor(0.0230)
tensor(0.0248)
tensor(0.0354)
tensor(0.0283)
tensor(0.0230)
tensor(0.0044)
tensor(0.0292)
tensor(0.0328)
tensor(0.0195)
tensor(0.0080)


In [63]:
GU = GaussianUncertainty("PhotoTourism")

In [65]:
icp.compute_cal_distance()

In [72]:
trans_distance = icp.trans_distances
trans_distance

tensor([4.0677, 4.3157, 4.4442,  ..., 7.4146, 4.0011, 4.0294],
       dtype=torch.float64)

In [88]:
trans_prune = np.percentile(trans_distance, 90)
trans_distance1 = trans_distance[trans_distance<=trans_prune]
trans_distance1.shape, trans_distance.shape

masks = (trans_distance<=trans_prune)
icp.gt, icp.pred, icp.gt_rot, icp.gt_trans, icp.pred_rot, icp.pred_trans = icp.gt[masks], icp.pred[masks], icp.gt_rot[masks], icp.gt_trans[masks], icp.pred_rot[masks], icp.pred_trans[masks]

In [89]:
icp.compute_nc_scores()
nc_scores = icp.compute_non_conformity_scores()['Trans']

nc_scores.shape


torch.Size([1016])

In [103]:
for i in range(len(test_poses)):
    p_valuess = []
    for j in range(len(trans_distance1)):
        test_nc_score = icp.trans_err(test_est_trans[i].repeat(len(icp.gt_trans), 1), icp.gt_trans)
        cal_nc_score = icp.trans_err(icp.gt_trans[j].repeat(len(icp.gt_trans), 1), icp.gt_trans)
        p_values = (test_cal_nc_score <= cal_nc_score).sum()
        p_valuess.append(p_values)
    print(torch.tensor(p_valuess).max())
        

tensor(99)
tensor(99)
tensor(99)
tensor(99)
tensor(99)
tensor(99)
tensor(99)
tensor(99)
tensor(99)
tensor(99)


In [98]:
trans_distance1.sum()

tensor(5055.0940, dtype=torch.float64)